In [16]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
drive.mount('/content/drive')
file_path = '/content/drive/My Drive/Data_sets/spam.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
try:
    df = pd.read_csv(file_path, encoding='latin-1', usecols=['v1', 'v2'])
    print("✅ File loaded successfully!")
except FileNotFoundError:
    print(f"❌ Error: Could not find file at {file_path}")
    print("Please ensure 'spam.csv' is in the top folder of your Google Drive.")
    # Stop execution if file isn't found
    raise

✅ File loaded successfully!


In [18]:
df.columns = ['label', 'message']


df['label_mapped'] = df['label'].map({'spam': 'Suspicious', 'ham': 'Not suspicious'})

print("\n--- Data Snapshot ---")
print(df.head())
print(f"\nTotal Messages: {len(df)}")


--- Data Snapshot ---
  label                                            message    label_mapped
0   ham  Go until jurong point, crazy.. Available only ...  Not suspicious
1   ham                      Ok lar... Joking wif u oni...  Not suspicious
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...      Suspicious
3   ham  U dun say so early hor... U c already then say...  Not suspicious
4   ham  Nah I don't think he goes to usf, he lives aro...  Not suspicious

Total Messages: 5572


In [19]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['message'])
y = df['label_mapped']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [20]:
y_pred = model.predict(X_test)

print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
cm = confusion_matrix(y_test, y_pred, labels=['Not suspicious', 'Suspicious'])
tn, fp, fn, tp = cm.ravel()

print("\n--- Detailed Analysis ---")
print(f"1. Suspicious messages correctly identified (True Positives): {tp}")
print(f"2. Genuine messages incorrectly flagged (False Positives):    {fp}")


--- Model Evaluation ---
Accuracy: 98.03%

--- Detailed Analysis ---
1. Suspicious messages correctly identified (True Positives): 139
2. Genuine messages incorrectly flagged (False Positives):    11


In [21]:
print("\n--- Testing Custom Messages ---")
custom_msgs = [
    "URGENT! You have won a 1 week FREE membership prize.",
    "Hey, are we still going for lunch tomorrow?"
]
custom_vec = vectorizer.transform(custom_msgs)
predictions = model.predict(custom_vec)

for msg, pred in zip(custom_msgs, predictions):
    print(f"Message: '{msg}'\nPrediction: {pred}\n")


--- Testing Custom Messages ---
Message: 'URGENT! You have won a 1 week FREE membership prize.'
Prediction: Suspicious

Message: 'Hey, are we still going for lunch tomorrow?'
Prediction: Not suspicious

